# Configuracion Roles
### Previamente los roles deben ser creados desde el account, esto no funciona en Databricks Free Edicion, pero se deja el codigo

## 1 Asinacion  de permisos a los Grupos

In [0]:
# ============================================
# Creación de grupos/roles de acceso
# Proyecto: fintech_finpay
# ============================================

catalog = "fintech_finpay"

roles = {
    "ingenieria": {
        "schemas": ["bronze", "silver", "gold", "observability"],
        "permissions": ["USE SCHEMA", "CREATE TABLE", "MODIFY"]
    },
    "riesgo": {
        "schemas": ["silver", "gold"],
        "permissions": ["USE SCHEMA", "SELECT"]
    },
    "auditoria": {
        "schemas": ["gold", "observability"],
        "permissions": ["USE SCHEMA", "SELECT"]
    }
}

# ============================================
# Asignar USE CATALOG
# ============================================

print("\n====================================")
print("Asignando USE CATALOG")
print("====================================")

for role in roles.keys():

    try:

        sql_use= f"""
        GRANT USE CATALOG
        ON CATALOG `{catalog}`
        TO `{role}`
        """
        spark.sql(sql_use)

        print(f"USE CATALOG otorgado a {role}")

    except Exception as e:

        print(f"Error asignando USE CATALOG a {role}")
        print(f"Sentencia usada {sql_use}")
        print(str(e))
# ============================================
# Asignar permisos por schema
# ============================================

print("\n====================================")
print("Asignando permisos por schema")
print("====================================")

for role, config in roles.items():

    for schema in config["schemas"]:

        for permission in config["permissions"]:

            try:

                spark.sql(f"""
                GRANT {permission}
                ON SCHEMA `{catalog}`.`{schema}`
                TO `{role}`
                """)

                print(
                    f"{permission} otorgado sobre "
                    f"{catalog}.{schema} a {role}"
                )

            except Exception as e:

                print(
                    f"Error asignando {permission} "
                    f"sobre {catalog}.{schema} a {role}"
                )

                print(str(e))

print("\n====================================")
print("Proceso finalizado.")
print("====================================")

## 2. Creacion de FUncion MAsk

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION fintech_finpay.silver.mask_pii_string(value STRING)
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN value
    ELSE '***MASKED***'
  END
""")

## 3. Asignacion de reow Level y mask a al columna

In [ ]:
spark.sql("""
ALTER TABLE fintech_finpay.silver.users
ALTER COLUMN full_name
SET MASK fintech_finpay.silver.mask_pii_string
""")

spark.sql("""
ALTER TABLE fintech_finpay.silver.users
ALTER COLUMN document_id
SET MASK fintech_finpay.silver.mask_pii_string
""")

spark.sql("""
ALTER TABLE fintech_finpay.silver.users
ALTER COLUMN email
SET MASK fintech_finpay.silver.mask_pii_string
""")

spark.sql("""
ALTER TABLE fintech_finpay.silver.users
ALTER COLUMN phone
SET MASK fintech_finpay.silver.mask_pii_string
""")

print("Column masks aplicadas correctamente.")